###Python’s built-in data structures (Lists, Tuples, Dictionaries, and Sets) are highly versatile, but they have inherent architectural and performance trade-offs.
Here are the primary limitations of each built-in structure:
### 1. Lists (list)
- Slow Arbitrary Insertions and Deletions: Inserting or deleting elements at the beginning or middle of a list takes \(O(n)\) time because Python must shift all subsequent elements in memory.
- Memory Overhead: Lists over-allocate memory slots to allow for fast \(O(1)\) appends. Because they store pointers to objects rather than the raw data itself, they consume significantly more memory than native arrays.
- Search Inefficiency: Finding an element in an unsorted list requires a linear search (\(O(n)\) time).
### 2. Tuples (tuple)
- Immutability: You cannot modify, add, or remove elements after creation. While this ensures data integrity, it requires creating an entirely new tuple to update data.
- Shallow Immutability: If a tuple contains mutable objects (like a list), the contents of that list can still be modified, which can cause subtle bugs.
### 3. Dictionaries (dict)
- Key Constraints: Keys must be hashable (immutable objects like strings, numbers, or tuples). You cannot use lists or other dictionaries as keys.
- Memory Footprint: Dictionaries are highly optimized hash tables but still carry a massive memory overhead compared to sequential data structures.
- No Duplicate Keys: Dictionaries inherently overwrite values if a duplicate key is assigned.
### 4. Sets (set)
- Element Constraints: Like dictionary keys, set elements must be hashable. You cannot have a set of lists.
- Unordered Nature: Sets do not maintain insertion order (unlike Python 3.7+ dictionaries) and do not support indexing or slicing (\(O(1)\) access by index is impossible).
- No Duplicates: Sets automatically discard duplicate values, making them unusable for datasets where frequency matters.

### Comparison Matrix

| Data Structure | Primary Strength | Major Limitation | Search Time | Memory Overhead |
| :--- | :--- | :--- | :--- | :--- |
| **List** | Ordered, dynamic resizing | Slow shifts on insert/delete \(O(n)\) | \(O(n)\) | Medium-High |
| **Tuple** | Immutable, memory-efficient | Cannot be altered after creation | \(O(n)\) | Low |
| **Dict** | Lightning-fast key lookups | Keys must be hashable | \(O(1)\) | High |
| **Set** | Uniqueness, mathematical operations | Unindexed, elements must be hashable | \(O(1)\) | High |



##List Insertion Bottleneck (\(O(n)\) Shift)
This script below demonstrates why lists are slow when inserting elements at the beginning versus the end. Inserting at the beginning forces Python to shift every single pointer in memory.

In [0]:
import time

# Create a large list
n = 100000
my_list = list(range(n))

# Measure inserting at the END (Fast: O(1))
start_time = time.time()
for _ in range(1000):
    my_list.append(None)
    my_list.pop()
print(f"Time to append/pop at the END: {time.time() - start_time:.5f} seconds")

# Measure inserting at the BEGINNING (Slow: O(n))
start_time = time.time()
for _ in range(1000):
    my_list.insert(0, None)
    my_list.pop(0)
print(f"Time to insert/pop at the BEGINNING: {time.time() - start_time:.5f} seconds")



##Dictionary / Set Key Constraints (Hashability)
This example highlights the architectural limitation where mutable structures (like lists) cannot be used as dictionary keys or set elements.

In [0]:
# Valid: Using an immutable tuple as a key
valid_dict = { (1, 2): "Coordinate A" }
print("Tuple key works:", valid_dict[(1, 2)])

try:
    # Valid: Attempting to use a mutable list as a value
    valid_dict2 = {"Coordinate B":[1, 2]}
    print("trying to use a mutable list as a value works:", valid_dict2["Coordinate B"])
except TypeError as e:
    print(f"\nDictionary Limitation Triggered: {e}")

try:
    # Invalid: Attempting to use a mutable list as a key
    invalid_dict = {[1, 2]: "Coordinate B"}
    print("trying to use a mutable list as a key works:", invalid_dict[[1, 2]])
except TypeError as e:
    print(f"\nDictionary Limitation Triggered: {e}")    

try:
    # Invalid: Attempting to put a list into a set
    invalid_set = { [1, 2, 3] }
except TypeError as e:
    print(f"Set Limitation Triggered: {e}")


##Tuple "Shallow" Immutability
This snippet proves that while a tuple itself cannot be altered, any mutable objects nested inside it can still be modified, risking silent data mutations.

In [0]:
# A tuple containing an integer and a mutable list
my_tuple = (42, ["apple", "banana"])
print("Original Tuple:", my_tuple)

try:
    # This will fail (Tuples are immutable)
    my_tuple[0] = 99
except TypeError as e:
    print(f"\nTuple protected from direct reassignment: {e}")

# This will SUCCESSFULY modify the data inside the tuple!
my_tuple[1].append("chery")
print("Mutated Tuple (Shallow Immutability):", my_tuple)
my_tuple[1][2]="cherry"
print("Mutated Tuple (Shallow Immutability):", my_tuple)

##Memory Footprint: Built-in vs. Optimized Array
This script showcases the massive memory overhead of standard Python lists (which hold pointers) compared to a contiguous block of raw data using Python's built-in array module.

|Feature|Python list|Python array module|numpy.array|
| :--- | :--- | :--- | :--- |
|**Source**|Built-in (Default)|Built-in (import array)|Third-party (import numpy)|
|**Data Types**|Can mix anything (strings, ints, dicts)|Must be all same type (e.g., all int)|Must be all same type (e.g., all int64)|
|**Memory Allocation|Array of pointers scattered in memory|Contiguous block of raw data	|Contiguous block of raw data|
|**Math Speed**|Slow (requires loops)|Slow (requires loops)	|Blazing fast (vectorized operations)|


To understand why this makes the built-in array module and numpy.array so fast and memory-efficient, it helps to contrast them with how a standard Python list handles memory.
### The Contrast: Python List vs. Contiguous Array
> 1. Standard Python List (An Array of Pointers)
>> A Python list does not actually store your data inside the list container. Instead, it stores memory addresses (pointers) that point to objects scattered all over your RAM.
>>> • How it looks in memory: If you have [10, 20, 30], the list is actually [Pointer A, Pointer B, Pointer C].
>>>> • The Problem:
>>>>> 	• Memory Overhead: Every single number is wrapped in a full Python object header (which takes 28 bytes for an integer), plus the 8 bytes for the pointer itself.
>>>>> 	• Cache Misses: Because the actual numbers are scattered randomly in RAM, your CPU has to constantly hop around to find the data. This slows down processing.

> 2. Python array and numpy.array (Contiguous Raw Data)
>> Both of these structures bypass Python's object wrapping. They reserve one single, uninterrupted block of memory and store the raw binary numbers right next to each other.
>>> • How it looks in memory: If you store [10, 20, 30] as 64-bit integers, the memory block allocates exactly 64 bits X 3 consecutively.
>>>> • The Benefit:
>>>>> 	• Zero Overhead: No pointers, no object wrappers. A 64-bit integer takes exactly 8 bytes of space.
>>>>> 	• CPU Cache Friendly: CPUs are designed to fetch data in blocks. When the CPU reads the first number, it automatically pre-loads the next numbers into its ultra-fast local cache because they are sitting right next to it.

###Memory Allocation: Python array vs. NumPy numpy.array
While both utilize contiguous memory blocks, how they manage and manipulate that memory is vastly different:
- Python array:       [ 10 ][ 20 ][ 30 ][ 40 ]  <-- Simple sequential block
- NumPy array:        [ 10 ][ 20 ][ 30 ][ 40 ]  <-- Raw Data Block
>                      Metadata: [ Shape=(2,2), Strides=(16, 8), Dtype=int64 ]
 
####Python Built-in array
- Simple Memory Management: It allocates a contiguous block of memory for 1D structural data.
- Resizing limitations: If you want to append an item and the block runs out of room, it must ask the operating system for a brand new, larger contiguous block, copy all the old data over, and free the old block.
- Strictly 1-Dimensional: It cannot natively represent matrices or grids without complex manual indexing math.

####NumPy Array (ndarray)
- Separation of Data and Metadata: NumPy separates the raw data block from the metadata (which includes the array's shape, data type, and "strides"—how many bytes to skip to get to the next row/column).
- Zero-Copy Views: Because the metadata is separate, you can reshape a 1D array into a 3D matrix instantly. NumPy does not copy or move the raw data in memory; it simply changes the metadata instructions on how to read that contiguous block.
- No Resizing by Design: NumPy arrays are strictly fixed-size upon creation. While np.append() exists, it creates an entirely new array under the hood. It trades away dynamic resizing to ensure the raw data block remains perfectly contiguous for massive mathematical operations.


In [0]:
import sys
import array
import numpy as np

# 1 million integers in a standard list
large_list = list(range(1_000_000))
# 1 million integers in a typed array (4-byte integers)
large_array = array.array('i', range(1_000_000))
# 1 million integers in a NumPy array (64-bit integers by default)
large_np_array = np.array(range(1_000_000))


#If you run sys.getsizeof(large_list), Python will only return the memory size of the list container itself (the array of pointers). It completely ignores the actual size of the objects (the integers) being pointed to.
# Pointers Container: ~8,000,056 bytes (~8 MB)
# 1 Million Integer Objects: 1,000,000 \(\times \)⁠ 28 bytes = 28,000,000 bytes (~28 MB)
# True Total List Memory: ~36 MB

# sys.getsizeof only measures the pointer array container for lists, 
# so we estimate total memory consumption.
list_memory = sys.getsizeof(large_list) + sum(sys.getsizeof(i) for i in large_list[:100]) * 10000
array_memory = sys.getsizeof(large_array)
np_memory = large_np_array.nbytes

print(f"Estimated List Memory:  {list_memory / (1024**2):.2f} MB")
print(f"Actual Typed Array Memory: {array_memory / (1024**2):.2f} MB")
print(f"NumPy Array Memory:           {np_memory / (1024**2):.2f} MB")
print(type(large_list))
print(type(large_array))
print(type(large_np_array))
print(large_list[:100])
print(large_array[:100])
print(large_np_array[:100])

In [0]:
import sys
import numpy as np
import array

# 1 million integers in a standard Python list
large_list = list(range(1_000_000))
# 1 million integers in a typed array (4-byte integers)
large_array = array.array('i', range(1_000_000))
# 1 million integers in a NumPy array (64-bit integers by default)
large_np_array = np.array(range(1_000_000))

# Calculate memory sizes
# For the list, we look at the structural size plus the elements
list_memory = sys.getsizeof(large_list) + sum(sys.getsizeof(i) for i in large_list[:100]) * 10000
np_memory = large_np_array.nbytes
array_memory = sys.getsizeof(large_array)

#If you run sys.getsizeof(large_list), Python will only return the memory size of the list container itself (the array of pointers). It completely ignores the actual size of the objects (the integers) being pointed to.
# Pointers Container: ~8,000,056 bytes (~8 MB)
# 1 Million Integer Objects: 1,000,000 \(\times \)⁠ 28 bytes = 28,000,000 bytes (~28 MB)
# True Total List Memory: ~36 MB
#Meanwhile, the NumPy array stores the raw data directly inside its container. Running large_np_array.nbytes gives you the exact, all-inclusive size right away, which is only 8 MB (8 bytes per 64-bit integer X⁠ 1 million).

print(f"Estimated Python List Memory: {list_memory / (1024**2):.2f} MB")
print(f"NumPy Array Memory:           {np_memory / (1024**2):.2f} MB")
print(f"Actual Typed Array Memory: {array_memory / (1024**2):.2f} MB")

In [0]:
import time
import numpy as np

# Setup data size (1 million items)
size = 1_000_000

# -------------------------------------------------------------
# 1. Standard Python Loop Approach
# -------------------------------------------------------------
python_list = list(range(size))

start_time = time.time()
# Process each element one by one using a loop
try:
    python_result = (python_list ** 2) + 10
except TypeError as e:
    print(f"\nERROR: {e}")
    python_result = []
    for x in python_list:
        python_result.append((x ** 2) + 10)
    
list_time = time.time() - start_time
print(f"Python Loop Execution Time:  {list_time:.5f} seconds")

# -------------------------------------------------------------
# 2. NumPy Vectorized Approach
# -------------------------------------------------------------
numpy_array = np.arange(size)

start_time = time.time()
# Process the entire block of memory simultaneously (Vectorization)
numpy_result = (numpy_array ** 2) + 10

numpy_time = time.time() - start_time
print(f"NumPy Vectorized Time:       {numpy_time:.5f} seconds")

# -------------------------------------------------------------
# Speedup Calculation
# -------------------------------------------------------------
print(f"\n🚀 NumPy is {list_time / numpy_time:.1f}x FASTER than standard Python loops!")
print(python_list[:10])
print(python_result[:10])
print(numpy_array[:10])
print(numpy_result[:10])

In [0]:
import numpy as np

def visualize_numpy_metadata(arr, name="Array"):
    print(f"=== {name} Metadata Visualization ===")
    
    # 1. Structural Dimensions
    print(f"Shape (Dimensions):      {arr.shape}")
    print(f"NDim (Number of axes):   {arr.ndim}")
    
    # 2. Data Type Properties
    print(f"Dtype (Data Type):       {arr.dtype}")
    print(f"Itemsize (Bytes/element): {arr.itemsize} bytes")
    
    # 3. Memory Layout & Layout Steps
    # Strides tells you how many bytes you must step in memory to move to the next item along each axis
    print(f"Strides (Bytes to step): {arr.strides}") 
    print(f"Total Bytes (nbytes):    {arr.nbytes} bytes")
    
    # 4. Underlying Memory Address Pointer
    # This exposes the raw computer memory address where the contiguous block starts
    raw_memory_address = arr.__array_interface__['data'][0]
    print(f"Base Memory Pointer:     {hex(raw_memory_address)}")
    
    # 5. Core Operational Flags
    print("\nMemory Layout Flags:")
    print(f"  - Is Contiguous in C-order (Row-major)?    {arr.flags['C_CONTIGUOUS']}")
    print(f"  - Is Contiguous in F-order (Column-major)? {arr.flags['F_CONTIGUOUS']}")
    print(f"  - Does it own its own raw memory block?    {arr.flags['OWNDATA']}")
    print("=" * 40 + "\n")

# -------------------------------------------------------------
# Demonstration: Creating a 1D Array and changing its Metadata
# -------------------------------------------------------------

# Create a flat 1D array of 4 integers (64-bit = 8 bytes each)
# Memory block allocated: 32 contiguous bytes total.
original_1d = np.array([10, 20, 30, 40], dtype=np.int64)
visualize_numpy_metadata(original_1d, "Original 1D Array")

# Create a "View" by reshaping it into a 2x2 matrix
# Crucial: This DOES NOT copy the data or change the raw memory block!
# It only creates a new metadata object pointing to the SAME raw address.
reshaped_2d = original_1d.reshape(2, 2)
visualize_numpy_metadata(reshaped_2d, "Reshaped 2D View")
print(original_1d)
print(reshaped_2d)

While NumPy is designed for ultra-fast math on a single machine using contiguous local memory, Spark is designed for Distributed Computing. It allows you to process datasets that are far too large to fit into the RAM of any single computer by splitting the data across a cluster of machines.

###The Architecture: NumPy vs. PySpark DataFrames
To understand Spark DataFrames, we must look at how they manage memory and execution compared to local tools:
|Feature|NumPy Array|PySpark DataFrame|
| :--- | :--- | :--- |
|Execution Context|Single Machine (Local CPU/RAM)	|Distributed Cluster (Driver + Multiple Executors)|
|Memory Strategy|Single Contiguous Memory Block	|Fragmented Partitions spread across cluster nodes|
|Evaluation Type|Eager Evaluation (Math happens immediately)	|Lazy Evaluation (Builds a plan, executes only when needed)|
|Scale Limit|Limited by the RAM of your single computer	|Scalable to Petabytes (Scale out by adding more machines)|

- Each machine (Worker Node) in your Databricks cluster processes its assigned partitions in parallel. If a machine runs out of RAM, Spark can spill individual partitions to disk instead of crashing with an OutOfMemory error like Python or NumPy would.


In [0]:
from pyspark.sql import functions as F
df = spark.range(0, 1_000_000)
df.limit(5).display()
df.show(2)
print(df.head(3))
print(df.take(4))
print(type(df)) 
df.printSchema()
print(df.schema.fields[0].dataType)

In [0]:
import time
import numpy as np
from pyspark.sql import functions as F
# -------------------------------------------------------------
# 1. Spark df Approach
# -------------------------------------------------------------
df = spark.range(0, 1_000_000)
start_time = time.time()
# Process each element one by one using a loop
try:
    df_result = (df ** 2) + 10
except TypeError as e:
    print(f"\nERROR: {e}")
    result_df = df.withColumn("id_times_10_plus_1", (F.col("id") ** 2) + 10)
    
list_time = time.time() - start_time

print(f"Python Loop Execution Time:  {list_time:.5f} seconds")
result_df.limit(5).display()

In [0]:
import time
import numpy as np
from pyspark.sql import functions as F
# Setup data size (1 million items)
size = 1_000_000

# -------------------------------------------------------------
# 1. Standard Python Loop Approach
# -------------------------------------------------------------
python_list = list(range(size))

start_time = time.time()
# Process each element one by one using a loop
python_result = []
for x in python_list:
    python_result.append((x ** 2) + 10)
    
list_time = time.time() - start_time
print(f"Python Loop Execution Time:  {list_time:.5f} seconds")

# -------------------------------------------------------------
# 2. NumPy Vectorized Approach
# -------------------------------------------------------------
numpy_array = np.arange(size)

start_time = time.time()
# Process the entire block of memory simultaneously (Vectorization)
numpy_result = (numpy_array ** 2) + 10
numpy_time = time.time() - start_time
print(f"NumPy Vectorized Time:       {numpy_time:.5f} seconds")
# -------------------------------------------------------------
# Speedup Calculation
# -------------------------------------------------------------
print(f"\n🚀 NumPy is {list_time / numpy_time:.1f}x FASTER than standard Python loops!")

# -------------------------------------------------------------
# 1. Spark df Approach
# -------------------------------------------------------------
df = spark.range(0, 1_000_000)
start_time = time.time()
# Process each element one by one using a loop
df = df.withColumn("id", (F.col("id") ** 2) + 10)  
df_time = time.time() - start_time
print(f"Spark df Execution Time:  {df_time:.5f} seconds")
# -------------------------------------------------------------
# Speedup Calculation
# -------------------------------------------------------------
print(f"\n🚀 NumPy is {df_time / numpy_time:.1f}x FASTER than standard Spark DF!")
df.limit(5).display()
# Row count (triggers a distributed action)
row_count = df.count()

# Column: id -> LongType (bigint) = 8 bytes of raw data per row
bytes_per_row = sum(f.dataType.simpleString() and {"bigint": 8, "long": 8, "int": 4, "double": 8, "string": None}.get(f.dataType.simpleString(), None) or 0 for f in df.schema.fields)
raw_bytes = row_count * bytes_per_row

print(f"Rows: {row_count:,}")
print(f"Columns: {[f.name for f in df.schema.fields]}, types: {[f.dataType.simpleString() for f in df.schema.fields]}")
print(f"Raw data estimate: {row_count:,} rows x {bytes_per_row} bytes = {raw_bytes:,} bytes = {raw_bytes / (1024**2):.2f} MiB")
print(f"Cached? {df.storageLevel}")

In [0]:
from pyspark.sql import functions as F

# 1. Generate a large distributed DataFrame (10 million numbers)
# This creates a Spark DataFrame instantly because of Lazy Evaluation.
# Spark hasn't actually generated the numbers yet; it just wrote down the plan to do so.
df = spark.range(0, 10_000_000)

# 2. Inspect Distributed Metadata (Partitions)
# Let's see how many chunks Spark split this dataset into across our cluster
num_partitions = df.rdd.getNumPartitions()
print(f"Initial Number of Partitions: {num_partitions}")

# 3. Apply Transformations (Lazy Evaluation)
# We are doing mathematical operations similar to our NumPy test.
# Notice how this cell runs in milliseconds—Spark is still just building the execution plan!
transformed_df = df.withColumn("squared_plus_ten", (F.col("id") ** 2) + 10)

# 4. View the Execution Plan
# This prints the logical and physical plan showing how the cluster will execute the query.
print("\n=== Spark Optimized Execution Plan ===")
transformed_df.explain(True)

# 5. Trigger an "Action" (Eager Execution)
# The data is only processed NOW because we asked for an actual output (.show() or .count()).
# Spark will now spin up the cluster tasks across all partitions simultaneously.
print("\nExecuting processing and fetching sample results...")
transformed_df.show(5)

**_While Lists, Tuples, Dictionaries, and Sets are Python's "core four" primitives, they leave massive operational gaps. In production engineering, data science, and big data, a few other data structures are considered absolute industry standards._**

The highly popular structural additions are divided into two main categories: Python standard library abstractions and big data/analytical abstractions.

**1. Standard Library Enhancements (Built-in but Specialized)**
Python includes a built-in module called collections specifically designed to solve the limitations of the core four data structures:
- collections.deque (Double-Ended Queue):
> 	• The Problem It Solves: Standard Python lists are incredibly slow (O(n)) when inserting or deleting elements from the beginning of the list.
> 	• The Structure: A deque is implemented as a doubly-linked list. It allows (O(1))⁠ lightning-fast appends and pops from both the front and the back, making it the industry standard for queues, streams, and sliding windows.
- collections.namedtuple:
> 	• The Problem It Solves: Standard tuples require you to access data using numeric indices (e.g., user[0], user[1]), making code unreadable and prone to bugs if the data order changes.
> 	• The Structure: It creates a tuple that allows you to access elements by named attributes (e.g., user.id, user.email), while retaining the low memory footprint and immutability of a standard tuple.
- collections.Counter:
> 	• The Problem It Solves: Finding the frequencies of items in a list using a standard dictionary requires boilerplate loops and conditional checks.
> 	• The Structure: A specialized dictionary subclass designed for counting hashable objects. You hand it a collection, and it instantly builds a frequency mapping with highly optimized helper methods like .most_common(n).

**2. Advanced & Computer Science Primitives**
When dealing with algorithms, complex relationships, or nested systems, structural data is organized into non-linear configurations:
- Graphs:
> 	• The Structure: A collection of Nodes (Vertices) connected by Edges.
> 	• Why It’s Popular: It is the foundational structure powering social media networks (friends connecting to friends), Google Maps routing (intersections connecting to streets), and recommendation engines. In Python, it is typically built using dictionaries or specialized packages like NetworkX.
- Trees / Heaps:
> 	• The Structure: Hierarchical parent-child nodes (like a family tree). A Heap is a specialized tree that allows you to instantly extract the absolute minimum or maximum value in a dataset in \(O(1)\)⁠ time. Python handles this natively via the heapq module.

**3. Analytical & Tabular Frameworks (Data Science)**
If you move into engineering data inside Databricks, Jupyter, or analytics pipelines, data is almost exclusively shifted into high-performance tabular objects:
- Pandas / Polars DataFrame:
> 	• The Structure: A 2D table-like structure with labeled rows and columns, heavily optimized for single-node data manipulation. It bridges the gap between structured SQL-like operations and programmatic Python.
- Spark StructType:
> 	• The Structure: In PySpark, this is the structural framework used to define a schema for nested, semi-structured records (like complex JSON payloads). It defines exact data types for multi-layered structures, allowing Spark to parse huge datasets uniformly across a cluster.

In [0]:
from collections import deque, namedtuple, Counter

# 1. High-Performance Queue (collections.deque)
# Unlike a list, popping from the left is O(1) instead of O(n)
print("=== 1. collections.deque ===")
queue = deque(["UserA", "UserB", "UserC"])
queue.append("UserD")       # Add to the right
queue.appendleft("Userz")       # Add to the left
print(f"Queue state: {list(queue)}\n")
queue.popleft()            # Remove from the left (Fast!)
print(f"Queue state: {list(queue)}\n")

# 2. Readable Immutable Data (collections.namedtuple)
print("=== 2. collections.namedtuple ===")
UserRecord = namedtuple('UserRecord', ['id', 'username', 'role'])
current_user = UserRecord(id=101, username="alice_dev", role="Admin")

# You can access elements by name instead of index!
print(f"User: {current_user.username} | Role: {current_user.role}")
print(f"Still works as a tuple by index: {current_user[0]}\n")

# 3. Frequency Tracker (collections.Counter)
print("=== 3. collections.Counter ===")
clickstream_logs = ["click", "view", "click", "purchase", "view", "click"]
log_counts = Counter(clickstream_logs)
print(f"Counts: {log_counts}")
print(f"Total Counts: {dict(log_counts)}")
print(f"Most Frequent Event: {log_counts.most_common(1)}")


In [0]:
import heapq

# 1. Graph Implementation (Adjacency List using dict)
print("=== 1. Graph (Network Nodes) ===")
# Representing a computer network: Node points to a list of connected nodes
network_graph = {
    "ServerA": ["ServerB", "DatabaseX"],
    "ServerB": ["ServerA", "Client1", "Client2"],
    "DatabaseX": ["ServerA"],
    "Client1": ["ServerB"],
    "Client2": ["ServerB"]
}
print(f"Nodes connected to ServerB: {network_graph['ServerB']}\n")

# 2. Min-Heap Priority Queue (heapq)
print("=== 2. Min-Heap (Priority Queue) ===")
# A heap keeps the absolute smallest value at index 0 at all times
data_stream = [50, 10, 40, 20]
print(f"Remaining Heap array: {data_stream}")
heapq.heapify(data_stream)  # Rearranges the list into a heap structure in-place

print(f"Initial Heap array: {data_stream}")
print(f"Smallest item is always at index 0: {data_stream[0]}")

# Push a new element and immediately pop the smallest remaining element
heapq.heappush(data_stream, 5)
print(f"Remaining Heap array: {data_stream}")
smallest = heapq.heappop(data_stream)
print(f"Extracted absolute smallest element: {smallest}")
print(f"Remaining Heap array: {data_stream}")


|Operation	|List (list)	|Deque (deque)	|Heap (heapq)|
| :--- | :--- | :--- | :--- |
|Insert / Delete at End	| O(1) (Amortized)	|O(1)	|(O(log n)) (To push/bubble up)|
|Insert / Delete at Front | O(n) (Slow! Shifts all items)	|O(1) (Blazing Fast)	|(O(log n)) (To pop and restore tree)|
|Random Access by Index	|O(1) (Instant lookup)	|O(n) (Must hop across blocks)	|O(1) for index 0; |O(n) for others*|
|Find Smallest Element	|O(n) (Must scan whole list)	|O(n) (Must scan whole deque)	|O(1) (Always at index 0)|

*Note: While a heap uses a list container under the hood allowing you to look at any index in O(1) time, looking at any index other than 0 yields unsorted data. To find a specific value or the next smallest value properly, you must structurally pop elements, which takes \(O(\log n)\) time.

In [0]:
%pip install polars

dbutils.library.restartPython()

In [0]:
# Note: Polars and PySpark come ready out-of-the-box in Databricks environments.
import polars as pl
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. High-Performance Local Tabular Data (Polars DataFrame)
print("=== 1. Polars DataFrame ===")
# Polars uses an underlying Arrow contiguous memory layout that is faster than Pandas
pl_df = pl.DataFrame({
    "Product": ["Laptop", "Mouse", "Monitor"],
    "Price": [1200, 25, 300]
})
print(pl_df.filter(pl.col("Price") > 50))
print("\n")

# 2. Schema Specification for Big Data (Spark StructType)
print("=== 2. Spark StructType Schema ===")
# Defining a precise metadata layout for an incoming corporate JSON or log file
user_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True)
])

# Creating sample data mapped to our structural schema
raw_data = [(1, "Alex", "Engineering"), (2, "Taylor", "Marketing")]
spark_df = spark.createDataFrame(raw_data, schema=user_schema)

print("Enforced Spark Schema Architecture:")
spark_df.printSchema()
spark_df.show()